In [1]:
!pip install -q langchain langchain-community langchain-openai langgraph groq

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

/home/imil/.pyenv/versions/Hugging/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# --- Validate API key early ---
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("❌ GROQ_API_KEY not found in .env")

# --- LLM Setup (Groq) ---
llm = ChatOpenAI(
    temperature=0,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b"
)




# --- Prompt 1: Extract ---
prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications (CPU, RAM, Storage) clearly from the text:\n\n{text_input}"
)

# --- Prompt 2: Force strict JSON ---
prompt_transform = ChatPromptTemplate.from_template(
    """Convert the following specifications into STRICT JSON.

Rules:
- Only return JSON
- Keys must be: cpu, memory, storage
- No explanation, no extra text

Specifications:
{specifications}
"""
)





# --- Chains ---
extraction_chain = prompt_extract | llm | StrOutputParser()

full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | llm
    | StrOutputParser()
)




# --- Run ---
input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."

result = full_chain.invoke({"text_input": input_text})

print("\n--- Final JSON Output ---")
print(result)


--- Final JSON Output ---
{
  "cpu": "3.5 GHz octa-core processor",
  "memory": "16 GB",
  "storage": "1 TB NVMe SSD"
}
